In [ ]:
# Select 25 pilot notes (with treatments, not in the inspection sample) and export them for Label Studio
import json
import random
from pathlib import Path

from datasets import load_dataset

ds = load_dataset("AGBonnet/augmented-clinical-notes", split="train")

random.seed(42)
inspected = set(random.sample(range(len(ds)), 40))


def has_treatment(i):
    try:
        s = json.loads(ds[i]["summary"], strict=False)
    except json.JSONDecodeError:
        return False
    tr = s.get("treatments")
    return isinstance(tr, list) and any(isinstance(t, dict) for t in tr)


rng = random.Random(7)
candidates = [i for i in rng.sample(range(len(ds)), 400) if i not in inspected]
pilot = [i for i in candidates if has_treatment(i)][:25]

tasks = [{"data": {"text": ds[i]["full_note"], "note_index": i}} for i in pilot]
out = Path("../data/labeling/pilot_tasks.json")
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(tasks, indent=2), encoding="utf-8")
print(len(tasks), "tasks written to", out.resolve())
print("note indexes:", pilot)